In [ ]:
from IPython.display import display
from PIL import Image
from transformers import set_seed

from common.io import find_project_root
from common.model import extract_answer, load_model, run_model

## Load the model

In [ ]:
project_root = find_project_root()
set_seed(42, deterministic=True)

model_config = {
    "path" : "JZPeterPan/MedVLM-R1",
    "hf_cache" : "/data/huggingface_cache"
}

model, processor, generation_config = load_model(model_config)

## Load VQA examples

In [ ]:
all_questions = [
    {
        "image": [project_root / "data/OmniMedVQA/sample_mri/images/mod-mri_000031.png"],
        "problem": "What type of diagnostic tool produced this image? A)Angiography, B)Endoscopy, C)X-ray, D)MRI",
        "solution": "D",
        "answer": "MRI"
    },
    {
        "image": [project_root / "result/MedVLM-R1/debug_bias_field_attack/attacked_image/attacked_image_1000.png"],
        "problem": "What type of diagnostic tool produced this image? A)Angiography, B)Endoscopy, C)X-ray, D)MRI",
        "solution": "D",
        "answer": "MRI"
    },
]

## Run inference 

In [ ]:
for i, question in enumerate(all_questions):

    output_text = run_model(
        question=question["problem"],
        image=question["image"][0],
        model=model,
        processor=processor,
        generation_config=generation_config
    )
    
    display(Image.open(question["image"][0]))
    
    print(question["problem"])
    
    model_answer = extract_answer(output_text, tag="answer")
    model_thought = extract_answer(output_text, tag="think")
    
    print(f"Model Answer: \033[94m{model_answer}\033[0m")
    print(f"Model Thought: \033[94m{model_thought}\033[0m")
    print(f"Groundtruth Solution: \033[92m{question['solution']}\033[0m")
        